# Fitted loan preprocessing

In [2]:
from pathlib import Path
import sys

In [3]:
import pandas as pd
from IPython.display import display
from ml_gearbox import audit as ga

In [4]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data/raw/data.csv"
assert RAW_PATH.is_file(), f"Missing raw data: {RAW_PATH}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
import joblib
import numpy as np

In [6]:
from src.data import get_or_create_split
from src.features import build_features
from src.preprocessing import audit_preprocessing, make_preprocessor
from src.schema import TARGET, ROW_ID

In [7]:
df = pd.read_csv(RAW_PATH)
split = get_or_create_split(df, path=PROJECT_ROOT / "data/processed/split.parquet")
train = split["train"]
holdout = split["test"]


In [8]:
train_features = build_features(train.drop(columns=TARGET))
holdout_features = build_features(holdout.drop(columns=TARGET))
preprocessing_audit = audit_preprocessing(train_features)
display(preprocessing_audit["missingness_after_cleaning"])
display(preprocessing_audit["dti_validation"])

,feature,missing_count,percentage
0,emp_length_years,28727,6.922769
1,revol_util,336,0.080971
2,dti,293,0.070609
3,loan_to_income,280,0.067476
4,payment_to_income,280,0.067476
5,inq_last_6mths,1,0.000241


,feature,check,violation_count,violation_pct
0,dti,min >= 0,1,0.0


Negative DTI and infinite values become missing. The preprocessor learns training medians, adds indicators for columns missing during training, and standardizes the resulting columns. All-missing training columns receive zero. High valid values remain unchanged before scaling.

Scaling is enabled here to demonstrate a reusable preparation step. The random forest in notebook 05 uses the same preprocessing without scaling.

In [9]:
prep = make_preprocessor(scale=True)
prepared_train = prep.fit_transform(train_features)
prepared_holdout = prep.transform(holdout_features)

display(ga.missingness_report(prepared_train))
display(ga.missingness_report(prepared_holdout))

,feature,missing_count,percentage


,feature,missing_count,percentage


In [10]:
assert prepared_train.shape[0] == len(train)
assert prepared_holdout.shape[0] == len(holdout)
assert prepared_train.columns.equals(prepared_holdout.columns)
assert prepared_train.index.equals(train.index)
assert prepared_holdout.index.equals(holdout.index)
assert TARGET not in prepared_train and ROW_ID not in prepared_train
assert np.isfinite(prepared_train.to_numpy()).all()
assert np.isfinite(prepared_holdout.to_numpy()).all()

display(pd.DataFrame({
    "rows": [prepared_train.shape[0], prepared_holdout.shape[0]],
    "features": [prepared_train.shape[1], prepared_holdout.shape[1]],
}, index=["training", "holdout"]))

,rows,features
training,414964,25
holdout,103742,25


Save the fitted preprocessor. To transform new raw predictors, call `build_features(raw_predictors)` before calling `bundle["preprocessor"].transform(...)`. Notebook 05 saves the complete model pipeline separately.

In [11]:
artifact_path = PROJECT_ROOT / "models/preparation.joblib"
artifact_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"preprocessor": prep}, artifact_path)
print(artifact_path)

c:\Users\USER\Desktop\Portfolio\supervised\ml-anomaly-detection\models\preparation.joblib
